# maskAnnotate GUI — Step-by-step debug notebook
Run each cell to test the components that `run_gui.py` assembles.

## 0. Imports & environment check

In [1]:
import os, sys

# Auto-detect Qt plugin path for conda environments (fixes cocoa missing on macOS)
_env_plugins = os.path.join(sys.prefix, "plugins")
if os.path.isdir(_env_plugins) and "QT_PLUGIN_PATH" not in os.environ:
    os.environ["QT_PLUGIN_PATH"] = _env_plugins

import numpy as np
print('numpy:', np.__version__)

import xarray as xr
print('xarray:', xr.__version__)

import scipy
print('scipy:', scipy.__version__)

import napari
print('napari:', napari.__version__)

import qtpy
print('Qt backend:', qtpy.API_NAME)

print('\nEnvironment OK')

numpy: 1.24.4
xarray: 0.16.2
scipy: 1.9.3
napari: 0.4.18
Qt backend: PyQt5

Environment OK


## 1. Test DataManager — load stacks and mask

In [2]:
from mask_annotate.data_manager import DataManager

dm = DataManager()

# Load sample data — adjust paths if needed
name1, shape1 = dm.load_stack('sample/stackMC.nc')
print(f'Loaded stack: {name1}, shape: {shape1}')

name2, shape2 = dm.load_stack('sample/dffStack.nc')
print(f'Loaded stack: {name2}, shape: {shape2}')

mask_shape = dm.load_mask('sample/roiMask.npy')
print(f'Loaded mask, shape: {mask_shape}')
print(f'Mask unique labels: {np.unique(dm.mask)}')

Loaded stack: stackMC, shape: (10800, 9, 180, 180)
Loaded stack: dffStack, shape: (10800, 9, 180, 180)
Loaded mask, shape: (9, 180, 180)
Mask unique labels: [-1  0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22
 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46
 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70
 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86]


## 2. Test aggregation (time-average)

In [3]:
# This may take a moment for large stacks
agg_mean = dm.get_aggregate('stackMC', 'mean')
print(f'Mean aggregate shape: {agg_mean.shape}, dtype: {agg_mean.dtype}')

agg_std = dm.get_aggregate('stackMC', 'std')
print(f'Std aggregate shape: {agg_std.shape}')

# Second call should be instant (cached)
agg_mean2 = dm.get_aggregate('stackMC', 'mean')
print(f'Cache hit: {agg_mean2 is agg_mean}')

Mean aggregate shape: (9, 180, 180), dtype: float32
Std aggregate shape: (9, 180, 180)
Cache hit: True


## 3. Test single timepoint extraction

In [4]:
tp = dm.get_timepoint('stackMC', 0)
print(f'Timepoint 0 shape: {tp.shape}, dtype: {tp.dtype}')
print(f'Number of timepoints: {dm.get_n_timepoints("stackMC")}')
print(f'Number of planes: {dm.get_n_planes("stackMC")}')

Timepoint 0 shape: (9, 180, 180), dtype: float32
Number of timepoints: 10800
Number of planes: 9


## 4. Launch napari and test ViewerManager

In [5]:
viewer = napari.Viewer(title='maskAnnotate debug')

In [6]:
from mask_annotate.viewer_manager import ViewerManager

vm = ViewerManager(viewer)

# Show the mean-aggregated reference image
vm.show_image(agg_mean, name='reference')
print('Image layer added')

# Overlay the mask as labels
vm.show_labels(dm.mask, name='mask')
print('Labels layer added — try editing with napari paint/erase tools')

Image layer added
Labels layer added — try editing with napari paint/erase tools


In [7]:
# Switch reference to std aggregation
vm.show_image(agg_std, name='reference')
print('Switched reference to std aggregate')

Switched reference to std aggregate


In [8]:
# Read back the labels (reflects any manual edits you made in napari)
edited_mask = vm.get_labels_data('mask')
print(f'Labels data shape: {edited_mask.shape}, unique: {np.unique(edited_mask)}')

Labels data shape: (9, 180, 180), unique: [-1  0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22
 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46
 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70
 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86]


## 5. Test ShiftModel

In [9]:
from mask_annotate.shift_model import ShiftModel

n_t = dm.get_n_timepoints('stackMC')
sm = ShiftModel(dm.mask, n_t)
print(f'ShiftModel: {sm.n_timepoints} timepoints, {sm.n_planes} planes')
print(f'Shifts array shape: {sm.shifts.shape}')

ShiftModel: 10800 timepoints, 9 planes
Shifts array shape: (10800, 9, 2)


In [10]:
# Apply no shift — should be identical to base mask
no_shift = sm.apply_shifts_for_timepoint(0)
print(f'No-shift matches base: {np.array_equal(no_shift, dm.mask)}')

No-shift matches base: True


In [11]:
# Apply a shift of (dx=3, dy=5) to plane 0 at timepoint 0
sm.set_shift(0, 0, dx=3, dy=5)
shifted = sm.apply_shifts_for_timepoint(0)

print(f'Shifted mask shape: {shifted.shape}')
print(f'Plane 0 changed: {not np.array_equal(shifted[0], dm.mask[0])}')
print(f'Plane 1 unchanged: {np.array_equal(shifted[1], dm.mask[1])}')

# Verify boundary handling — shifted-out pixels should be 0
print(f'Top-left corner of shifted plane 0 (should be 0): {shifted[0, :5, :3]}')

Shifted mask shape: (9, 180, 180)
Plane 0 changed: True
Plane 1 unchanged: True
Top-left corner of shifted plane 0 (should be 0): [[0 0 0]
 [0 0 0]
 [0 0 0]
 [0 0 0]
 [0 0 0]]


In [12]:
# Visualize the shift in napari
vm.clear()
img_t0 = dm.get_timepoint('stackMC', 0)
vm.show_image(img_t0, name='reference')
vm.show_labels(shifted, name='mask')
print('Showing timepoint 0 with shifted mask — compare plane 0 vs other planes')

Showing timepoint 0 with shifted mask — compare plane 0 vs other planes


## 6. Test the full widget docking (Stage 1 → 2 → 3)

In [13]:
# Close previous viewer and start fresh
viewer.close()

viewer = napari.Viewer(title='maskAnnotate — full GUI test')
dm2 = DataManager()
vm2 = ViewerManager(viewer)

from mask_annotate.load_widget import LoadWidget
from mask_annotate.cleanup_widget import CleanupWidget
from mask_annotate.shift_widget import ShiftWidget
from qtpy.QtWidgets import QTabBar
from qtpy.QtCore import QTimer

load_w = LoadWidget(dm2)
cleanup_w = CleanupWidget(dm2, vm2)
shift_w = ShiftWidget(dm2, vm2)

# Dock each widget individually, then tabify
dock_load = viewer.window.add_dock_widget(load_w, name='1. Load', area='right')
dock_cleanup = viewer.window.add_dock_widget(cleanup_w, name='2. Cleanup', area='right')
dock_shift = viewer.window.add_dock_widget(shift_w, name='3. Shift', area='right')

viewer.window._qt_window.tabifyDockWidget(dock_load, dock_cleanup)
viewer.window._qt_window.tabifyDockWidget(dock_cleanup, dock_shift)
dock_load.raise_()

# Wire tab switches — deferred so QTabBar is fully constructed
prev_tab = [0]

def _on_tab_changed(index, tab_bar):
    prev = prev_tab[0]
    if prev == index:
        return
    prev_text = tab_bar.tabText(prev)
    new_text = tab_bar.tabText(index)
    if "Load" in prev_text:
        load_w.apply_roi_filter()
    elif "Cleanup" in prev_text:
        cleanup_w._sync_labels_to_data()
    if "Cleanup" in new_text:
        cleanup_w.activate()
    elif "Shift" in new_text:
        shift_w.activate()
    prev_tab[0] = index

def _connect_tab_bar():
    qt_window = viewer.window._qt_window
    for tb in qt_window.findChildren(QTabBar):
        tab_texts = [tb.tabText(i) for i in range(tb.count())]
        if any("Load" in t for t in tab_texts):
            tb.currentChanged.connect(lambda idx, _tb=tb: _on_tab_changed(idx, _tb))
            print(f'Connected to QTabBar with tabs: {tab_texts}')
            break
    else:
        print('WARNING: Could not find QTabBar with Load tab')

QTimer.singleShot(0, _connect_tab_bar)

print('GUI launched — use the Load tab to add stacks and mask, then click tabs to switch stages')

GUI launched — use the Load tab to add stacks and mask, then click tabs to switch stages


In [14]:
import numpy as np
mask = np.load('sample/roiMask.npy')
labels = np.unique(mask)
print(f'Unique labels: {len(labels)}')
print(f'Min: {labels.min()}, Max: {labels.max()}')
print(f'Labels: {labels}')


Connected to QTabBar with tabs: ['1. Load', '2. Cleanup', '3. Shift']
Unique labels: 88
Min: -1.0, Max: 86.0
Labels: [-1.  0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16.
 17. 18. 19. 20. 21. 22. 23. 24. 25. 26. 27. 28. 29. 30. 31. 32. 33. 34.
 35. 36. 37. 38. 39. 40. 41. 42. 43. 44. 45. 46. 47. 48. 49. 50. 51. 52.
 53. 54. 55. 56. 57. 58. 59. 60. 61. 62. 63. 64. 65. 66. 67. 68. 69. 70.
 71. 72. 73. 74. 75. 76. 77. 78. 79. 80. 81. 82. 83. 84. 85. 86.]
